In [2]:
import numpy as np
from concurrent.futures import ThreadPoolExecutor

MAX_ITER = 30
AMBIENT_DARKNESS = 0.8  # Environmental factor (0.0 = bright, 1.0 = pitch dark)


def get_user_inputs():
    # 1. Take grid dimension N (Rows == Columns)
    while True:
        try:
            n = int(input("Enter number of rows/columns (N): "))
            if n > 0:
                break
            print("Please enter a positive integer.")
        except ValueError:
            print("Invalid input. Enter a valid integer.")

    # 2. Take N x N movement grid values between 0 and 1
    print(f"\nEnter the {n}x{n} movement grid (space-separated values between 0 and 1):")
    movement_grid = np.zeros((n, n))

    for r in range(n):
        while True:
            try:
                row_values = list(map(float, input(f"Row {r + 1}: ").split()))
                if len(row_values) != n:
                    print(f"Error: Expected {n} values, got {len(row_values)}. Try again.")
                    continue
                if any(val < 0.0 or val > 1.0 for val in row_values):
                    print("Error: All values must be between 0.0 and 1.0. Try again.")
                    continue
                movement_grid[r] = row_values
                break
            except ValueError:
                print("Invalid input. Enter numeric values separated by spaces.")

    return n, movement_grid


def evaluate_fitness(intensity, req_light):
    energy_cost = intensity
    under_lit_penalty = max(0.0, req_light - intensity) * 5.0
    over_lit_penalty = max(0.0, intensity - req_light) * 1.5
    return -(energy_cost + under_lit_penalty + over_lit_penalty)


def get_neighbors(n, r, c):
    neighbors = []
    for dr in [-1, 0, 1]:
        for dc in [-1, 0, 1]:
            if dr == 0 and dc == 0:
                continue
            nr, nc = r + dr, c + dc
            if 0 <= nr < n and 0 <= nc < n:
                neighbors.append((nr, nc))
    return neighbors


def update_cell(args):
    r, c, n, current_grid, fitness_grid, movement_grid, req_illum = args
    neighbors = get_neighbors(n, r, c)

    if neighbors:
        best_nr, best_nc = max(neighbors, key=lambda pos: fitness_grid[pos])
        neighbor_movement = np.mean([movement_grid[nr, nc] for nr, nc in neighbors])
        best_neighbor_intensity = current_grid[best_nr, best_nc]
    else:
        # Fallback if N == 1 (single cell with no neighbors)
        neighbor_movement = movement_grid[r, c]
        best_neighbor_intensity = current_grid[r, c]

    target = 0.6 * req_illum[r, c] + 0.4 * neighbor_movement
    candidate_intensity = (
        0.5 * current_grid[r, c]
        + 0.3 * target
        + 0.2 * best_neighbor_intensity
    )
    candidate_intensity = np.clip(candidate_intensity, 0.2, 1.0)

    new_fit = evaluate_fitness(candidate_intensity, req_illum[r, c])
    if new_fit > fitness_grid[r, c]:
        return r, c, candidate_intensity, new_fit
    return r, c, current_grid[r, c], fitness_grid[r, c]


def run_pca():
    n, movement_grid = get_user_inputs()

    # Calculate required illumination based on movement and environmental darkness
    req_illum = np.clip(movement_grid * AMBIENT_DARKNESS + 0.15, 0.2, 1.0)

    # Initialize streetlights at high brightness (0.7 to 1.0)
    np.random.seed(42)
    intensity_grid = np.random.uniform(0.7, 1.0, (n, n))
    initial_energy = np.sum(intensity_grid)

    # Initial parallel fitness evaluation
    fitness_grid = np.zeros((n, n))
    for r in range(n):
        for c in range(n):
            fitness_grid[r, c] = evaluate_fitness(intensity_grid[r, c], req_illum[r, c])

    # Parallel Cellular Optimization Loop
    with ThreadPoolExecutor() as executor:
        for _ in range(MAX_ITER):
            next_grid = np.copy(intensity_grid)
            next_fitness = np.copy(fitness_grid)

            tasks = [
                (r, c, n, intensity_grid, fitness_grid, movement_grid, req_illum)
                for r in range(n)
                for c in range(n)
            ]
            results = executor.map(update_cell, tasks)

            # Synchronize states
            for r, c, new_val, new_fit in results:
                next_grid[r, c] = new_val
                next_fitness[r, c] = new_fit

            intensity_grid = next_grid
            fitness_grid = next_fitness

    final_energy = np.sum(intensity_grid)
    energy_saved = ((initial_energy - final_energy) / initial_energy) * 100

    print("\n--- Results ---")
    print("Input Movement Grid:\n", np.round(movement_grid, 2))
    print("\nOptimized Streetlight Intensity Grid:\n", np.round(intensity_grid, 2))
    print(f"\nInitial Total Power: {initial_energy:.2f}")
    print(f"Optimized Total Power: {final_energy:.2f}")
    print(f"Energy Saved: {energy_saved:.2f}%")


if __name__ == "__main__":
    run_pca()


Enter number of rows/columns (N): 3

Enter the 3x3 movement grid (space-separated values between 0 and 1):
Row 1: 0.1 0.2 0.1
Row 2: 0.2 0.9 0.3
Row 3: 0.1 0.2 0.1

--- Results ---
Input Movement Grid:
 [[0.1 0.2 0.1]
 [0.2 0.9 0.3]
 [0.1 0.2 0.1]]

Optimized Streetlight Intensity Grid:
 [[0.31 0.31 0.32]
 [0.31 0.75 0.4 ]
 [0.31 0.31 0.32]]

Initial Total Power: 7.65
Optimized Total Power: 3.34
Energy Saved: 56.30%
